# Worked Example: Assembling Analysis DataFrames

## Goal
Build a tidy long dataframe (participant, electrode, trial, time, power, regressors)
from sample feedback epochs — the input shape expected by `statistics_utils`.


In [ ]:
import numpy as np
import pandas as pd
import mne
from pathlib import Path

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
epochs = mne.read_epochs(Path('../../data/sample_feedback_start-epo.fif'), preload=True, verbose=False)
epochs.metadata = beh[['reward', 'rpe']].copy()
epochs.metadata['trial'] = np.arange(len(epochs))

elec_df = pd.read_csv(Path('../../data/sample_labels_bp'))
chan = 'racas1-racas2'
roi = elec_df.loc[elec_df.label == chan, 'salman_region'].iloc[0]

ep = epochs.copy().pick([chan]).filter(13, 30, verbose=False)
times = ep.times
step = max(1, len(times) // 20)
rows = []
for t_idx in range(0, len(times), step):
    power = ep.get_data()[:, 0, t_idx] ** 2
    for trial_i, p in enumerate(power):
        rows.append({
            'participant': 'sample',
            'unique_label': chan,
            'roi': roi,
            'trial': trial_i,
            'ts': times[t_idx],
            'tfr': p,
            'rpe': epochs.metadata['rpe'].iloc[trial_i],
            'reward': epochs.metadata['reward'].iloc[trial_i],
        })
smoothed_df = pd.DataFrame(rows)
smoothed_df.head()

In [ ]:
print('shape:', smoothed_df.shape)
print('n timepoints:', smoothed_df['ts'].nunique())
print('n trials:', smoothed_df['trial'].nunique())
print(smoothed_df.groupby('ts')['tfr'].mean().head())

## Next step

Chapter 14 (`14_group_level_statistics`) covers group-level statistics.